# Imputacion De Datos Faltantes

Importamos las librerias y datasets necesarios para el trabajo

In [4]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import missingno as msno
import pandas as pd
import seaborn as sns
import numpy as np

from src.pandas_accessors import MissingMethods
from src.data_loader import load_all_datasets

datasets = load_all_datasets()

riskfactors =datasets['riskfactors']

print(f"Librerias y datasets: {datasets.keys()} Cargados correctamente")

Librerias y datasets: dict_keys(['oceanbuoys', 'pedestrian', 'riskfactors', 'pima']) Cargados correctamente


### Importaciones extra de paquetes mas especificos para el momento de imputar los datos

Aca importaremos las siguientes librerias:

* janitor
* nhanes
* scipy
* session_info
* sklearn
* statsmodels

In [3]:
import janitor
import nhanes
import scipy.stats
import session_info

import sklearn.compose
import sklearn.impute
import sklearn.preprocessing

import statsmodels.api as sm
import statsmodels.datasets
import statsmodels.formula.api as smf
from statsmodels.graphics.mosaicplot import mosaic

from sklearn.ensemble import RandomForestRegressor
from sklearn.experimental import enable_iterative_imputer
from sklearn.kernel_approximation import Nystroem
from sklearn.linear_model import BayesianRidge, Ridge
from sklearn.neighbors import KNeighborsRegressor

print("Librerias cargadas correctamente")

Librerias cargadas correctamente


### Configurar aspecto general de las graficas del proyecto

De esta manera todas mantendras el mismo aspect ratio y estilo

In [6]:
%matplotlib inline

sns.set_theme(
    style="whitegrid",
    rc={
        "figure.figsize": (8, 6),
    }
)


### El problema de trabajar con valores faltantes:

En este caso python por default con la mayoria de valores faltantes que se registran cono ***nan*** les hace un proceso de eliminacion por lo que no son tomados en cuenta para el calculo en especifico que sea hace

Hay que recordar que python por default lo que hace es una elminacion en el proceso del calculo lo que hace que un calculo se haga bajo mas filas que otras y altere de manera completa el resultado ya que unas se pueden evaluar con 5 filas y otro resultado con 8

Por eso se recomienda hacer la imputacion

In [12]:
airquality_df = (
    sm.datasets.get_rdataset("airquality")
    .data
    .clean_names(
        case_type="snake",
    )
    .add_column("year", 1973)
    .assign(
        date = lambda df: pd.to_datetime(df [["year", "month", "day"]])
    )
    .sort_values(by = "date")
    .set_index("date")
)
print("Dataset airquality cargado correctamente")
airquality_df

Dataset airquality cargado correctamente


/home/jefred/anaconda3/envs/datosFaltantes/lib/python3.14/site-packages/pandas_flavor/register.py:161: FutureWarning: This function will be deprecated in a 1.x release. Please use `pd.DataFrame.assign` instead.
  return method(self._obj, *args, **kwargs)


,ozone,solar_r,wind,temp,month,day,year
date,,,,,,,
1973-05-01,41.0,190.0,7.4,67,5,1,1973
1973-05-02,36.0,118.0,8.0,72,5,2,1973
1973-05-03,12.0,149.0,12.6,74,5,3,1973
1973-05-04,18.0,313.0,11.5,62,5,4,1973
1973-05-05,NaN,NaN,14.3,56,5,5,1973
...,...,...,...,...,...,...,...
1973-09-26,30.0,193.0,6.9,70,9,26,1973
1973-09-27,NaN,145.0,13.2,77,9,27,1973
1973-09-28,14.0,191.0,14.3,75,9,28,1973


In [18]:
print("El numero de observaciones es de 116 en este calculo, a pesar de que el dataset tiene 153 observaciones,\n esto es debido a que hay 37 observaciones con valores faltantes en la variable ozone,\n por lo que se descartan para el calculo de la regresion lineal")

(
    smf.ols(
        formula = "temp ~ ozone",
        data=airquality_df
    )
    .fit()
    .summary()
    .tables[0]
)



El numero de observaciones es de 116 en este calculo, a pesar de que el dataset tiene 153 observaciones,
 esto es debido a que hay 37 observaciones con valores faltantes en la variable ozone,
 por lo que se descartan para el calculo de la regresion lineal


Dep. Variable:,temp,R-squared:,0.488
Model:,OLS,Adj. R-squared:,0.483
Method:,Least Squares,F-statistic:,108.5
Date:,"Fri, 04 Sep 2026",Prob (F-statistic):,2.93e-18
Time:,16:00:04,Log-Likelihood:,-386.27
No. Observations:,116,AIC:,776.5
Df Residuals:,114,BIC:,782.1
Df Model:,1,,
Covariance Type:,nonrobust,,


In [20]:
print("El numero de observaciones es de 146 en este calculo, a pesar de que el dataset tiene 153 observaciones,\n esto es debido a que hay 7 observaciones con valores faltantes en la variable ozone,\n por lo que se descartan para el calculo de la regresion lineal")

(
    smf.ols(
        formula = "temp ~ solar_r",
        data=airquality_df
    )
    .fit()
    .summary()
    .tables[0]
)

El numero de observaciones es de 146 en este calculo, a pesar de que el dataset tiene 153 observaciones,
 esto es debido a que hay 7 observaciones con valores faltantes en la variable ozone,
 por lo que se descartan para el calculo de la regresion lineal


Dep. Variable:,temp,R-squared:,0.076
Model:,OLS,Adj. R-squared:,0.070
Method:,Least Squares,F-statistic:,11.86
Date:,"Fri, 04 Sep 2026",Prob (F-statistic):,0.000752
Time:,16:00:55,Log-Likelihood:,-525.28
No. Observations:,146,AIC:,1055.
Df Residuals:,144,BIC:,1061.
Df Model:,1,,
Covariance Type:,nonrobust,,


### Proceso De Analisis y limpieza de datos